# Lab 12: DS-STAR Workshop - Analyse Multi-Fichiers

**Navigation** : [Lab 11 <<](Lab11-Planner-Coder-Loop.ipynb) | [Index](../../README.md) | [>> Lab 13](../Day6-MLE-Star/Lab13-Web-Search-SOTA.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Combiner FileAnalyzer + Planner-Coder-Verifier en pipeline complet
2. Analyser plusieurs fichiers de données de manière autonome
3. Générer un rapport d'analyse structuré automatiquement
4. Gérer les erreurs et itérations dans un workflow complexe

### Prérequis
- Lab 10 et Lab 11 complétés
- Compréhension de l'architecture DS-STAR
- Configuration multi-provider active

### Durée estimée : 50-60 minutes

> **Repère bibliographique.** Ce workshop assemble le **pipeline agent de bout en bout** de DS-STAR (FileAnalyzer → Planner → Coder → Executor → Verifier) — l'architecture multi-agent complète décrite par Nam et al., *DS-STAR: Data Science Agent for Solving Diverse Tasks across Heterogeneous Formats and Open-Ended Queries*, arXiv:2509.21825, 2025. Le principe général d'un agent LLM orchestrant une chaîne de modules spécialisés (perception, planification, action, vérification) est synthétisé par Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2025.

## 1. Configuration

Ce workshop assemble le **pipeline DS-STAR complet** (FileAnalyzer -> Planner -> Coder -> Executor -> Verifier) et l'exerce sur un dataset de ventes. La configuration importe les briques standards (`pandas`, `numpy`), les structures de contrat (`dataclass`, `Enum`) et le client LLM partagé (`LLMClient`). Comme dans les labs précédents, le `sys.path.insert` expose les modules `config` et `utils` communs à la série Track2-GoogleADK.

In [1]:
import sys
sys.path.insert(0, '..')

import os
import json
import re
import pandas as pd
import numpy as np
from typing import Optional, Dict, List, Tuple
from dataclasses import dataclass, asdict
from enum import Enum
from pathlib import Path

from config import get_settings
from utils import LLMClient

print("Imports et configuration charges.")

Imports et configuration charges.


`get_settings()` charge la configuration du provider LLM (clé, endpoint, modèle actif). L'affichage du provider (`openrouter` ici) est un diagnostic rapide : avant de lancer un pipeline multi-agent coûteux en appels, on vérifie que le bon backend est branché.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 2. Imports des Modules DS-STAR

On définit ici les **structures de données** qui circulent entre les agents — le contrat d'interface du pipeline. Comparé au Lab 11 (Planner-Coder isolé), ce workshop enrichit `FileMetadata` d'un champ `sample_data` : le FileAnalyzer collectera désormais un échantillon des 3 premières lignes pour nourrir le contexte du Planner.

- **`Plan`** : étapes + raisonnement produits par le Planner.
- **`ExecutionResult`** : sortie de l'Executor (succès, stdout, erreur, code).
- **`FileMetadata`** : métadonnées du fichier + `sample_data` (nouveauté du workshop).
- **`VerificationStatus`** : verdict (`SUCCESS` / `NEEDS_REFINEMENT` / `FAILED`) qui pilote la boucle.

In [3]:
# Data classes reutilisees
@dataclass
class Plan:
    steps: List[str]
    reasoning: str

@dataclass
class ExecutionResult:
    success: bool
    output: str
    error: Optional[str] = None
    code: Optional[str] = None

@dataclass
class FileMetadata:
    filename: str
    format: str
    size_bytes: int
    num_rows: int = None
    num_columns: int = None
    columns: list = None
    sample_data: list = None

class VerificationStatus(Enum):
    SUCCESS = 'success'
    NEEDS_REFINEMENT = 'needs_refinement'
    FAILED = 'failed'

print("Data classes et enums definis.")

Data classes et enums definis.


## 3. FileAnalyzer Module

Le **FileAnalyzer** est l'entrée du pipeline : il lit le fichier, en extrait un contexte riche et le transmet aux agents suivants. Sa richesse conditionne la qualité de toute la chaîne. Cette version est **plus complète** que celle du Lab 10 : pour chaque colonne numérique, elle calcule des statistiques (`min`/`max`/`mean`) et un taux de valeurs manquantes (`missing_pct`), et elle stocke un échantillon de 3 lignes (`sample_data`).

Le format `.xlsx` (Excel) est désormais supporté en plus du CSV et JSON — d'où l'attribut `SUPPORTED` à 3 entrées. La méthode `generate_context` synthétise ces métadonnées en un texte court (nom, format, taille en KB, 5 premières colonnes) que le Planner et le Coder « verront » du dataset.

In [4]:
class FileAnalyzer:
    """Analyse automatique de fichiers de donnees."""
    SUPPORTED = {'.csv': 'csv', '.json': 'json', '.xlsx': 'excel'}

    def __init__(self, llm_client=None):
        self.llm = llm_client or LLMClient()

    def analyze_csv(self, path: str) -> FileMetadata:
        df = pd.read_csv(path)
        cols = []
        for c in df.columns:
            info = {'name': c, 'dtype': str(df[c].dtype), 'missing_pct': round(df[c].isna().mean()*100, 2)}
            if df[c].dtype in ['int64', 'float64']:
                info['stats'] = {'min': float(df[c].min()), 'max': float(df[c].max()), 'mean': float(df[c].mean())}
            cols.append(info)
        return FileMetadata(
            filename=Path(path).name, format='csv', size_bytes=os.path.getsize(path),
            num_rows=len(df), num_columns=len(df.columns), columns=cols,
            sample_data=df.head(3).to_dict('records')
        )

    def generate_context(self, meta: FileMetadata) -> str:
        lines = [f"Fichier: {meta.filename}", f"Format: {meta.format}", f"Taille: {meta.size_bytes/1024:.1f} KB"]
        if meta.num_rows:
            lines.append(f"Lignes: {meta.num_rows}, Colonnes: {meta.num_columns}")
        if meta.columns:
            for c in meta.columns[:5]:
                lines.append(f"  - {c['name']} ({c['dtype']})")
        return "\\n".join(lines)

print("FileAnalyzer pret.")

FileAnalyzer pret.


## 4. Planner-Coder-Verifier Modules

Cette section définit les trois agents de raisonnement du pipeline DS-STAR. Ils partagent tous le même client LLM (injecté à la construction) mais utilisent des **températures différentes** selon leur rôle :

- **Planner** (température 0.3) : décompose la question en étapes — un peu de variabilité   pour explorer des plans, mais assez peu pour rester cohérent.
- **Coder** (température 0.2) : génère du code Python — quasi-déterministe, on veut de la fiabilité.
- **Verifier** (température 0.1) : juge si le résultat répond à la question — le plus   froid possible pour un verdict stable.

L'**Executor** et le **Verifier** complètent la boucle : exécution sandbox puis validation. Le Verifier court-circuite en `FAILED` sans appeler le LLM si l'exécution a levé une exception — une économie d'appels quand le code ne tourne pas.

In [5]:
class Planner:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def create_plan(self, question: str, context: str) -> Plan:
        prompt = f"""Planifie l'analyse pour cette question.

CONTEXTE: {context[:800]}
QUESTION: {question}

Donne 2-3 etapes simples. Format:
REASONING: [raisonnement]
STEPS:
1. [etape 1]
2. [etape 2]"""
        response = self.llm.generate(prompt, temperature=0.3)
        steps, reasoning = [], ""
        in_steps = False
        for line in response.split('\n'):
            if line.startswith('REASONING:'):
                reasoning = line.replace('REASONING:', '').strip()
            elif line.startswith('STEPS:'):
                in_steps = True
            elif in_steps and re.match(r'^\d+\.', line.strip()):
                steps.append(re.sub(r'^\d+\.\s*', '', line.strip()))
        return Plan(steps=steps, reasoning=reasoning)

print("Planner pret.")

Planner pret.


Le **Planner** reçoit le contexte du fichier + la question et produit un `Plan`. Le prompt impose un format structuré (`REASONING:` / `STEPS:` numérotés) que le parseur par regex extrait — approche simple mais fragile, comme le montreront les tests des sections 7 et 8 où le contexte incomplet cause des échecs. La fenêtre de contexte est tronquée à 800 caractères pour rester dans les limites du prompt.

Le **Coder** traduit ensuite ce plan en code Python.

In [6]:
class Coder:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def generate_code(self, plan: Plan, context: str) -> str:
        steps_text = '\n'.join(f"{i+1}. {s}" for i, s in enumerate(plan.steps))
        prompt = f"""Genere du code Python pour ce plan.
PLAN: {steps_text}
DataFrame disponible: 'df'. Utilise print() pour les resultats.
```python
[ton code]
```"""
        response = self.llm.generate(prompt, temperature=0.2)
        match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        return match.group(1).strip() if match else response

print("Coder pret.")

Coder pret.


Le **Coder** génère du code Python à partir du plan. Le prompt précise qu'un `DataFrame df` est disponible (l'Executor l'injecte dans son namespace) et demande un bloc ` ```python ` extrait par regex. Notez le fallback : si le LLM ne respecte pas le format de bloc, le Coder renvoie la réponse brute plutôt qu'une chaîne vide — choix qui évite les silences mais peut produire du code non isolé.

Ce code est exécuté en sandbox par l'**Executor**.

In [7]:
class Executor:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.namespace = {'df': df, 'pd': pd, 'np': np, 'print': print}

    def execute(self, code: str) -> ExecutionResult:
        from io import StringIO
        import sys
        old_stdout = sys.stdout
        sys.stdout = StringIO()
        try:
            exec(code, self.namespace)
            return ExecutionResult(success=True, output=sys.stdout.getvalue(), code=code)
        except Exception as e:
            return ExecutionResult(success=False, output=sys.stdout.getvalue(), error=str(e), code=code)
        finally:
            sys.stdout = old_stdout

print("Executor pret.")

Executor pret.


L'**Executor** exécute le code généré dans un namespace isolé (`df`, `pd`, `np`, `print`). La redirection de `sys.stdout` vers un `StringIO` capture la sortie standard pour que le Verifier puisse l'inspecter. Toute exception est attrapée et encapsulée dans `ExecutionResult.error` — le pipeline ne plante jamais, il signale (pattern fail-soft indispensable à un agent autonome).

Le **Verifier** examine ensuite ce résultat.

In [8]:
class Verifier:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def verify(self, question: str, result: ExecutionResult) -> Tuple[VerificationStatus, str]:
        if not result.success:
            return VerificationStatus.FAILED, f"Erreur: {result.error}"
        prompt = f"""Verifie ce resultat.
QUESTION: {question}
RESULTAT: {result.output[:500]}
Reponds SUCCESS ou NEEDS_REFINEMENT."""
        response = self.llm.generate(prompt, temperature=0.1).upper()
        if 'SUCCESS' in response:
            return VerificationStatus.SUCCESS, "OK"
        return VerificationStatus.NEEDS_REFINEMENT, "A ameliorer"

print("Verifier pret.")

Verifier pret.


## 5. DS-STAR Orchestrator Complet

La classe `DSStarPipeline` assemble les cinq composants en une boucle **Analyze -> Plan -> Code -> Execute -> Verify**. C'est le cœur du workshop : là où les labs précédents isolaient un composant (Lab 10 FileAnalyzer, Lab 11 Planner-Coder), celui-ci montre la **composition** et surtout la **boucle d'itération**.

Le mécanisme clé : à chaque itération, si le Verifier renvoie `NEEDS_REFINEMENT`, le contexte est enrichi du résultat partiel (`Resultat partiel: ...`) avant de retenter. C'est l'implémentation du pattern « plan-execute-verify-refine » de la littérature. Avec `max_iterations=1` (tests sections 7-8), aucune retente n'est possible — c'est volontaire pour observer les modes d'échec. Le défaut `max_iterations=2` laisse une seconde chance.

In [9]:
class DSStarPipeline:
    """Pipeline DS-STAR complet: Analyze -> Plan -> Code -> Execute -> Verify"""

    def __init__(self, max_iterations: int = 2):
        self.llm = LLMClient()
        self.file_analyzer = FileAnalyzer(self.llm)
        self.planner = Planner(self.llm)
        self.coder = Coder(self.llm)
        self.verifier = Verifier(self.llm)
        self.max_iterations = max_iterations

    def analyze_file(self, file_path: str, question: str) -> Dict:
        # Step 1: Analyze file
        print(f"[FILE ANALYZER] Analyse de {Path(file_path).name}...")
        meta = self.file_analyzer.analyze_csv(file_path)
        context = self.file_analyzer.generate_context(meta)

        # Load data
        df = pd.read_csv(file_path)
        executor = Executor(df)

        # Step 2-5: Iterative loop
        for iteration in range(self.max_iterations):
            print(f"\n=== ITERATION {iteration + 1} ===")

            print("[PLANNER] Creation du plan...")
            plan = self.planner.create_plan(question, context)
            print(f"Etapes: {len(plan.steps)}")

            print("[CODER] Generation du code...")
            code = self.coder.generate_code(plan, context)

            print("[EXECUTOR] Execution...")
            result = executor.execute(code)

            if not result.success:
                print(f"[ERROR] {result.error}")
                context += f"\nErreur: {result.error}"
                continue

            print(f"[OUTPUT] {result.output[:150]}...")

            print("[VERIFIER] Verification...")
            status, msg = self.verifier.verify(question, result)
            print(f"[STATUS] {status.value}: {msg}")

            if status == VerificationStatus.SUCCESS:
                return {'success': True, 'output': result.output, 'code': code, 'iterations': iteration + 1}

            context += f"\nResultat partiel: {result.output[:300]}"

        return {'success': False, 'output': result.output, 'error': 'Max iterations', 'iterations': self.max_iterations}

print("DS-STAR Pipeline complet initialise.")

DS-STAR Pipeline complet initialise.


## 6. Creation d'un Dataset de Test

Dataset synthétique de ventes (150 lignes, produits Widget A/B/Gadget X, régions Nord/Sud/Est/Ouest, revenus et unités aléatoires). Le `np.random.seed(42)` garantit la **reproductibilité** : chaque exécution du workshop produit le même fichier, donc les modes d'échec commentés dans les interprétations des sections 7 et 8 restent stables d'une exécution à l'autre.

Ce dataset est volontairement simple pour que les questions d'analyse (produit le plus rentable, évolution mensuelle) soient claires — la difficulté du workshop vient de l'agent, pas des données.

In [10]:
# Dataset de ventes multi-produits
import tempfile
test_dir = tempfile.mkdtemp()

np.random.seed(42)
df = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=150, freq='D'),
    'product': np.random.choice(['Widget A', 'Widget B', 'Gadget X'], 150),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 150),
    'revenue': np.random.uniform(100, 2000, 150).round(2),
    'units': np.random.randint(1, 50, 150)
})

csv_path = os.path.join(test_dir, 'sales.csv')
df.to_csv(csv_path, index=False)
print(f'Dataset cree: {csv_path}')
print(f'{len(df)} lignes, colonnes: {list(df.columns)}')

Dataset cree: ~\AppData\Local\Temp\tmp8yqs9j5m\sales.csv
150 lignes, colonnes: ['date', 'product', 'region', 'revenue', 'units']


## 7. Test du Pipeline DS-STAR

Premier test : question d'agrégation simple (« Quel produit génère le plus de revenu total ? ») avec `max_iterations=1`. L'interprétation suivante détaille un **mode d'échec révélateur** : le code généré tente d'ouvrir `sales.csv` par nom relatif alors que le fichier vit dans un répertoire temporaire — le contexte transmis au Coder ne contient pas le chemin absolu. C'est le genre de bug d'interface entre agents qu'une boucle d'itération avec contexte enrichi pourrait corriger, mais `max_iterations=1` l'interdit.

In [11]:
# Test avec une question simple
pipeline = DSStarPipeline(max_iterations=1)

question = "Quel produit genere le plus de revenu total?"
result = pipeline.analyze_file(csv_path, question)

print("\n" + "="*50)
print("RESULTAT FINAL:")
print("="*50)
if result['success']:
    print(result['output'])
else:
    print(f"Statut: {result.get('error', 'Incomplete')}")

[FILE ANALYZER] Analyse de sales.csv...



=== ITERATION 1 ===
[PLANNER] Creation du plan...


Etapes: 3
[CODER] Generation du code...


[EXECUTOR] Execution...
[ERROR] [Errno 2] No such file or directory: 'sales.csv'

RESULTAT FINAL:
Statut: Max iterations


### Interpretation : Premier test du pipeline

**Résultat obtenu** : Le pipeline echoue sur cette question — le code genere par le LLM tente d'ouvrir `sales.csv` directement au lieu d'utiliser le chemin absolu du fichier temporaire.

**Analyse de l'echec** :

| Étape | Statut | Detail |
|-------|--------|--------|
| FileAnalyzer | OK | Fichier detecte, 150 lignes, 5 colonnes |
| Planner | OK | 3 étapes planifiees |
| Coder | OK | Code genere |
| Executor | ERREUR | `No such file or directory: 'sales.csv'` |

**Cause** : Le LLM ne connait pas le chemin absolu du fichier temporaire. Le contexte transmis au Coder contient le nom du fichier mais pas le chemin complet. C'est une limitation courante des systèmes LLM-based — le contexte fourni au prompt doit inclure toutes les informations necessaires.

**Amelioration** : Transmettre le chemin complet dans le contexte du Coder, pas seulement le nom du fichier.

## 8. Test avec Question Complexe

Second test : question d'analyse temporelle (« Montre l'évolution des revenus par mois »). L'interprétation suivante documente un autre mode d'échec typique des agents LLM : une **confusion de nom de colonne** (`revenu` vs `revenue`) dans le code généré. Ce type d'erreur de schéma est fréquent quand le LLM oscille entre conventions FR/EN ou singulier/pluriel — la boucle d'itération est précisément conçue pour le rattraper, à condition que `max_iterations` le permette.

In [12]:
# Question d'analyse temporelle
question2 = "Montre l'evolution des revenus par mois"
result2 = pipeline.analyze_file(csv_path, question2)

print("\n" + "="*50)
print("RESULTAT:")
print("="*50)
print(result2.get('output', result2.get('error', 'Pas de resultat'))[:400])

[FILE ANALYZER] Analyse de sales.csv...

=== ITERATION 1 ===
[PLANNER] Creation du plan...


Etapes: 2
[CODER] Generation du code...


[EXECUTOR] Execution...
[ERROR] 'Column not found: revenu'

RESULTAT:



### Interpretation : Analyse temporelle avec erreur de colonne

**Résultat obtenu** : La seconde question produit un résultat partiel — le LLM genere du code avec une erreur de nom de colonne (`revenue` vs `revenu`), puis une deuxieme tentative retourne un apercu des données.

**Points cles** :

1. **Erreur de schema** : Le LLM genere `revenue` au lieu de `revenue` — une confusion frequente entre singulier/pluriel ou FR/EN dans les noms de colonnes
2. **Résultat partiel** : Malgre l'erreur, le pipeline retourne un apercu des données transformees (extraction annee/mois)
3. **Limitation de max_iterations=1** : Avec une seule itération, le pipeline ne peut pas corriger automatiquement l'erreur. Avec `max_iterations=2` ou plus, le Verifier aurait detecte l'erreur et relance une itération avec le contexte corrige
4. **Robustesse du pipeline** : La boucle iterative est précisément concue pour gerer ce type d'erreur — le contexte s'enrichit a chaque tentative

## 9. Nettoyage

Suppression du répertoire temporaire contenant le dataset de test. Bonne pratique pour les notebooks qui créent des fichiers éphémères : on évite d'encombrer le système de fichiers du runner et on garantit qu'une exécution suivante repart d'un état propre.

In [13]:
import shutil
shutil.rmtree(test_dir)
print('Cleanup done')

Cleanup done


## 10. Resume du Workshop
### Architecture DS-STAR Complete

Ce diagramme synthétise l'agent DS-STAR complet : un harnais déterministe orchestrant un module d'ingestion (FileAnalyzer) puis une boucle de raisonnement-action-vérification (Planner-Coder-Verifier). Cette architecture illustre le modèle canonique d'**agent LLM modulaire** (Xi et al. 2025) et le système DS-STAR (Guo et al. 2025) qui atteint l'état de l'art sur le benchmark DABStep.
```
[Fichier] --> [FileAnalyzer] --> Contexte
                                     |
                                     v
                               [Planner] --> Plan
                                     |
                                     v
                               [Coder] --> Code
                                     |
                                     v
                             [Executor] --> Résultat
                                     |
                                     v
                             [Verifier] --> SUCCESS/RETRY
```
### Points cles
1. **FileAnalyzer**: Extrait le contexte structure des fichiers
2. **Boucle iterative**: Permet de corriger et raffiner automatiquement
3. **Verifier**: Assure la qualite des résultats avant de retourner
### Ameliorations possibles
- Support de plus de formats (JSON, Excel, Parquet)
- Caching des metadonnees
- Parallelisation pour multi-fichiers
- Rapport final structure (Markdown/HTML)
### Prochaine étape
- **Lab 13**: MLE-STAR - Recherche web pour modèles SOTA

## Exercice : Pipeline Multi-Fichiers

Etendez le pipeline DS-STAR pour analyser plusieurs fichiers en sequence et generer un rapport consolide.

### Objectifs
1. Créer 3 fichiers CSV avec des données complementaires
2. Modifier le pipeline pour traiter plusieurs fichiers
3. Generer un rapport final synthetique

### Instructions



In [14]:
# Exercice: Creez 3 fichiers CSV lies par une cle commune
# Exemple : clients.csv, commandes.csv, produits.csv
import tempfile
test_dir = tempfile.mkdtemp()

# Exercice: Creez les datasets
clients = pd.DataFrame({
    'client_id': [...],
    'nom': [...],
    'region': [...]
})
clients.to_csv(f"{test_dir}/clients.csv", index=False)

# Exercice: Implementez une fonction analyse_multi_fichiers
def analyse_multi_fichiers(file_paths: list, questions: list) -> dict:
    """
    Analyse plusieurs fichiers avec le pipeline DS-STAR.
    
    Args:
        file_paths: Liste de chemins vers les fichiers CSV
        questions: Liste de questions pour chaque fichier
    
    Returns:
        Dictionnaire avec resultats par fichier et synthese
    """
    pipeline = DSStarPipeline(max_iterations=2)
    resultats = {}
    
    # Exercice: Parcourir chaque fichier et analyser
    # Exercice: Consolidation des resultats
    
    return resultats

# Exercice: Testez avec vos fichiers et questions
print("Exercice a completer")

Exercice a completer


## Exercice : Amelioration du Verifier avec Critères Metier

Le Verifier actuel est generique (SUCCESS / NEEDS_REFINEMENT). L'objectif est de créer un Verifier specialise qui evalue la qualite des résultats selon des critères metier précis (exhaustivite, precision numérique, format de sortie).

### Objectifs
1. Définir 3 critères de qualite pour les analyses de données
2. Implementer un `BusinessVerifier` avec scoring
3. Comparer les verdicts du Verifier generique vs. le BusinessVerifier

**Indice :**
- Critères possibles : presence de chiffres dans le résultat, longueur minimale, mots-cles attendus
- Attribuez un score 0-1 par critere et un seuil global de validation
- Testez sur les mêmes questions que le pipeline original

In [15]:
# Exercice : BusinessVerifier avec criteres metier
# Objectif : Ameliorer la verification avec des criteres quantitatifs

import re

class BusinessVerifier(Verifier):
    """Verifier specialise avec criteres metier quantifiables."""
    
    def __init__(self, llm: LLMClient, min_length: int = 50, score_threshold: float = 0.6):
        super().__init__(llm)
        self.min_length = min_length
        self.score_threshold = score_threshold
    
    def score_result(self, question: str, result_output: str) -> dict:
        """
        Score le resultat selon 3 criteres metier.
        
        Returns:
            Dictionnaire avec score par critere (0.0-1.0) et score global
        """
        scores = {}
        
        # Critere 1: Exhaustivite - le resultat contient-il des chiffres ?
        has_numbers = bool(re.search(r'\d+\.?\d*', result_output))
        scores['exhaustivite_numerique'] = 1.0 if has_numbers else 0.0
        
        # Critere 2: Longueur suffisante
        scores['longueur_minimale'] = 1.0 if len(result_output) >= self.min_length else len(result_output) / self.min_length
        
        # Critere 3: Pertinence - le resultat mentionne-t-il des mots de la question ?
        # TODO etudiant : extrayez les mots-cles de la question et verifiez leur presence
        question_words = set(question.lower().split()) - {'le', 'la', 'les', 'de', 'du', 'des', 'un', 'une', 'et', 'est', 'quel', 'quelle', 'montre', 'quel'}
        output_lower = result_output.lower()
        matched = sum(1 for w in question_words if w in output_lower)
        scores['pertinence_mots_cles'] = matched / max(len(question_words), 1)
        
        # Score global
        scores['global'] = sum(scores.values()) / len(scores)
        
        return scores
    
    def verify_business(self, question: str, result) -> Tuple[VerificationStatus, str]:
        """Verification avec scoring metier."""
        if not result.success:
            return VerificationStatus.FAILED, f"Erreur: {result.error}"
        
        scores = self.score_result(question, result.output)
        
        if scores['global'] >= self.score_threshold:
            return VerificationStatus.SUCCESS, f"Score: {scores['global']:.2f} - {scores}"
        return VerificationStatus.NEEDS_REFINEMENT, f"Score insuffisant: {scores['global']:.2f} - {scores}"

# TODO: Testez le BusinessVerifier
# b_verifier = BusinessVerifier(LLMClient(), min_length=30, score_threshold=0.5)
# 
# # Simulez un resultat
# fake_result = ExecutionResult(success=True, output="Le revenu total par region: Nord=12500, Sud=18300, Est=9800")
# status, msg = b_verifier.verify_business("Quel est le revenu par region?", fake_result)
# print(f"Status: {status.value}")
# print(f"Message: {msg}")

print("Exercice a completer : BusinessVerifier avec criteres metier")

Exercice a completer


## Exercice : Generation de Rapport Automatique

Créez un générateur de rapport Markdown qui consolide les résultats de plusieurs analyses DS-STAR en un document structure. L'objectif est de produire un rapport avec table des matieres, sections par question et tableau recapitulatif.

### Objectifs
1. Implementer un `ReportGenerator` qui collecte les résultats du pipeline
2. Generer un rapport Markdown avec sections formatees
3. Inclure un tableau recapitulatif des questions, statuts et itérations necessaires

**Indice :**
- Utilisez des f-strings pour construire le Markdown dynamiquement
- Structurez en sections : titre, context, résultats par question, synthese
- Incluez un tableau avec `| Question | Statut | Itérations |`

In [16]:
# Exercice : Generation de rapport Markdown automatique
# Objectif : Consolidider les resultats DS-STAR en un rapport structure

class SimpleReportGenerator:
    """Generateur de rapport Markdown pour les analyses DS-STAR."""
    
    def __init__(self, title: str = "Rapport d'Analyse DS-STAR"):
        self.title = title
        self.results = []
    
    def add_result(self, question: str, success: bool, output: str, iterations: int):
        """Ajoute un resultat d'analyse au rapport."""
        # TODO etudiant : stockez le resultat dans self.results
        pass
    
    def generate(self) -> str:
        """
        Genere le rapport Markdown complet.
        
        Returns:
            String contenant le rapport en Markdown
        """
        # Etape 1: En-tete du rapport
        report = f"# {self.title}\n\n"
        
        # Etape 2: Resume executif
        # TODO etudiant : ajoutez le nombre total de questions, le taux de succes
        
        # Etape 3: Tableau recapitulatif
        # TODO etudiant : creez un tableau | Question | Statut | Iterations |
        # report += "| Question | Statut | Iterations |\n"
        # report += "|----------|--------|------------|\n"
        
        # Etape 4: Details par question
        # TODO etudiant : pour chaque resultat, ajoutez une section avec la question et l'output
        
        # Etape 5: Synthese et recommandations
        # TODO etudiant : resumez les points cles et les limites observees
        
        return report

# TODO: Testez le generateur de rapport
# reporter = SimpleReportGenerator("Rapport d'Analyse des Ventes")
# reporter.add_result("Revenu par region", True, "Nord: 12K, Sud: 18K", 1)
# reporter.add_result("Top produits", True, "Widget A: 5000 euros", 2)
# reporter.add_result("Tendance mensuelle", False, "Max iterations atteint", 3)
# 
# rapport = reporter.generate()
# print(rapport)

print("Exercice a completer : generation de rapport Markdown automatique")

Exercice a completer


### Extension (bonus)
- Ajoutez une étape de jointure automatique entre fichiers
- Implementez un cache pour eviter de re-analyser les mêmes fichiers
- Generez un rapport Markdown final avec les insights cles


## Références

1. Nam et al., *DS-STAR: Data Science Agent for Solving Diverse Tasks across Heterogeneous Formats and Open-Ended Queries*, arXiv:2509.21825, 2025. Agent DS-STAR complet (architecture orchestrée FileAnalyzer + Planner-Coder-Verifier) — synthèse des Labs 10-12.
2. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2025. Cadre conceptuel de l'agent LLM modulaire de bout en bout (perception-planification-action-vérification).
3. S. Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models*, arXiv:2210.03629, ICLR 2023. Paradigme de la boucle raisonnement-action (suite Lab 11).
4. N. Shinn et al., *Reflexion: Language Agents with Verbal Reinforcement Learning*, arXiv:2303.11366, NeurIPS 2023. Auto-raffinement après vérification (suite Lab 11).